# 第1回：データ分析の準備

この回は4つのパートで構成します：**Python環境とuv、Gitの基礎 ／ Pythonを読み、Copilotと少し変える ／ pandasで表データに触る ／ 分布・欠損・外れ値を確認する**。

**セルの動かし方**：各セル（灰色の枠）を選んで `Shift + Enter`（またはセル左の▷ボタン）を押すと実行できます。
**上から順に**実行してください。前のセルを飛ばすと、後のセルでエラーになります。

**AIと一緒に進める**：分からないコードは、セル全体ではなく気になる数行をM365 CopilotなどのAIへ貼り、
説明や修正を相談します。ただし、提案されたコードは必ず実行結果を見て確かめます。

まず「基本」と「演習」を進めます。「補足」は必要に応じて読み、
「発展（任意）」「追加演習（任意）」「自由課題（任意）」は飛ばしても構いません。


In [ ]:
# 【準備セル】教材フォルダの場所を自動で見つけます。中身は今は理解しなくてOK、そのまま実行してください。
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回で扱うこと

Python環境・uv・Gitの基礎を知り、Pythonとpandasの基本操作を身につけ、データの分布・欠損・外れ値を確認します。

### 進め方

この回は4つのパートに分かれています。パート1から順に「基本」と「演習」を進めてください。
1日で終える必要はありません。「発展（任意）」と「追加演習（任意）」は、余裕がある場合だけ取り組みます。

### 用語について

初めて出る用語は、その用語を使うセルで説明します。ここでまとめて暗記する必要はありません。

> **実行前の30秒予想**：各パートの問いに、今の言葉で仮の答えを書いてから始めます。


---

# パート1：Python環境とuv、Gitの基礎

**このパートの問い：自分のパソコンで、なぜ同じPythonの環境を再現できるのか。**


## 「環境」とは何か

Pythonを使うプロジェクトでは、プロジェクトごとに「使うPythonのバージョン」や「入れておく
ライブラリの種類・バージョン」が違います。あるプロジェクトはpandas 2.0を使い、別のプロジェクトは
pandas 1.5でないと動かない、ということも起こります。

そこで、プロジェクトごとにPythonとライブラリ一式を**別の場所に分けて用意**し、混ざらないようにします。
この「分けて用意した場所」を**仮想環境**と呼びます。この勉強会でいう`.venv`フォルダが、
この教材専用の仮想環境です。準備セルは、その仮想環境の中にあるPythonを実際に動かしています。


## 今、動いているPythonを確認する

`sys.executable`で、今のセルを実行しているPython本体がどこにあるかを確認できます。


In [ ]:
import sys
print("実行中のPython:", sys.executable)
print("バージョン:", sys.version.split()[0])


### 出力の読み方

`...\.venv\Scripts\python.exe`のように、勉強会フォルダの中の`.venv`を指していれば、正しい環境で
実行できています。VS Codeでカーネルを選ぶ操作は、「どの`python.exe`でこのNotebookを動かすか」を
選んでいる、というのがここでの確認でつながります。


## なぜuvを使うのか

`uv`は、仮想環境の作成と、必要なライブラリのインストールを両方行うツールです。この勉強会での
役割は次の2つです。

- `uv sync`：`pyproject.toml`と`uv.lock`を読み、`.venv`を作って必要なライブラリを入れる
- `uv run ...`：その`.venv`の中でコマンドを実行する（`uv run jupyter lab`など）

データサイエンス分野では、`uv`の他に**Conda**（Anaconda/Miniconda）もよく使われます。どちらも
「環境を分けて再現する」ためのツールですが、得意分野が少し違います。


| | uv | Conda |
|---|---|---|
| 主な対象 | Pythonのライブラリ | Python本体を含む、非Pythonのソフトウェアも扱える |
| 得意な場面 | 純粋なPythonプロジェクトを、速く・軽く構築する | RDKitのように、C/Fortranなどで書かれた部分を含む科学技術系ライブラリを扱う |
| この勉強会での位置づけ | 標準環境として採用 | 未使用（第4回のRDKitは`uv sync --extra chemistry`で導入） |

どちらが優れているというより、**プロジェクトの性質に合わせて選ぶもの**です。この勉強会は
構築のしやすさを優先して`uv`を選びましたが、「仮想環境を分けて再現する」という考え方自体は
どちらも同じです。


## 演習：ライブラリのバージョンを確認する

`pyproject.toml`と`uv.lock`には、使うライブラリのバージョンが記録されています。
`importlib.metadata`で、今の環境に実際に入っているバージョンを確認しましょう。


In [ ]:
from importlib.metadata import version

for library in ["pandas", "scikit-learn", "matplotlib"]:
    print(f"{library}: {version(library)}")


### 出力の読み方

ここに出たバージョンは、`uv.lock`に固定された、**この勉強会の全員が同じ値になる**バージョンです。
別の人のパソコンで同じセルを実行しても、同じ数字が出るはずです。これが「環境を再現する」の意味です。


## Gitとは何か（概念紹介）

Gitは、ファイルの変更履歴を記録し、後から見返したり元に戻したりできるようにする
**バージョン管理システム**です。この勉強会のリポジトリ自体もGitで管理され、GitHub上で公開されています。

ただし、この勉強会に**Gitの操作は必須ではありません**。ZIPダウンロードだけで最後まで完結します。
ここでは、用語だけ知っておきましょう。

- **リポジトリ（repository）**：ファイルとその変更履歴をまとめて保存する場所
- **コミット（commit）**：「ここまでの変更」に名前（メッセージ）を付けて記録する操作
- **クローン（clone）**：リポジトリを丸ごと自分のPCへコピーすること（ZIPダウンロードに近いが、履歴も含めてコピーされ、後から`pull`で更新できる）
- **プル（pull）**：リポジトリの最新の変更を、自分のPCへ取り込むこと
- **プッシュ（push）**：自分のPCで行った変更を、リポジトリ側へ反映すること
- **ブランチ（branch）**：同じリポジトリの中で、複数の変更を並行して進めるための分岐


## 演習：ターミナルでuvのバージョンを確認する

このNotebookの外、VS Codeの**ターミナル**で次を実行してみましょう（このセルではなく、ターミナルで実行します）。

```powershell
uv --version
```

バージョン番号が表示されれば、`uv`が正しくインストールされています。Gitを実際に試してみたい人は、
[Python環境とGitの基礎（任意）](../../docs/environment-and-git-basics.md)の手順でclone/pullを体験できます。


## まとめ

- プロジェクトごとに**仮想環境**を分けることで、ライブラリのバージョン衝突を避けられる。
- `uv`は仮想環境の作成とライブラリのインストールを行うツール。Condaは非Pythonの依存も扱える点が違う。
- `uv.lock`があるおかげで、**誰のパソコンでも同じバージョン**が再現される。
- Gitは変更履歴を管理する仕組みだが、この勉強会では必須ではない。


## 発展（任意）：Gitでの共同作業の考え方

ここからは、Gitをチームで使う場面をもう少し詳しく知りたい人向けの発展です。実際に手を動かす
必要はありません。


### ブランチとPull Request

複数人が同じリポジトリを同時に変更すると、作業がぶつかります。そこで、それぞれが**ブランチ**という
分岐を作り、自分の変更をそこで進めます。

作業が終わったら、**Pull Request（PR）**という形で「このブランチの変更を、本流（`main`）へ
取り込んでほしい」と提案します。他の人がレビューし、問題なければ**マージ（merge）**して統合します。
この「分岐して、レビューして、統合する」流れが、Gitがチーム開発で広く使われる理由です。


### リモートとローカル

自分のPC上のリポジトリを**ローカル**、GitHub上のリポジトリを**リモート**と呼びます。`clone`は
リモートをローカルへコピーする操作、`pull`はリモートの最新をローカルへ取り込む操作、`push`は
ローカルの変更をリモートへ反映する操作です。この勉強会のように「読むだけ」であれば`clone`と`pull`
だけで足り、`push`（自分の変更を反映する操作）は使いません。


## 追加演習（任意）

ここから先は90分では扱いません。手を動かして深めたい人向けの追加コードです。飛ばして次回へ進んでも
問題ありません。


### pyproject.tomlの中身を実際に読む

`pyproject.toml`は、このプロジェクトが使うライブラリを宣言するファイルです。テキストファイルなので、
Pythonからそのまま読めます。


In [ ]:
pyproject_text = (ROOT / "pyproject.toml").read_text(encoding="utf-8")
print(pyproject_text[:600])


### 出力の読み方

`dependencies = [...]`のあたりに、`pandas`や`scikit-learn`などのライブラリ名とバージョン条件が
並んでいるはずです。`uv sync`は、このファイルと`uv.lock`を読んで`.venv`を組み立てています。


### インストール済みライブラリの数を数える

`importlib.metadata`で、今の仮想環境に入っている全ライブラリの数も数えられます。


In [ ]:
from importlib.metadata import distributions

installed = sorted(d.metadata["Name"] for d in distributions())
print(f"インストール済み: {len(installed)}個")
print(installed[:10])


### 出力の読み方

直接使っているライブラリ（pandasなど）だけでなく、それが依存する別のライブラリも一緒に
インストールされているため、見た目より多い数になります。`uv.lock`は、この**全部の組み合わせ**を
固定しているファイルです。


---

# パート2：Pythonを読み、Copilotと少し変える

**このパートの問い：分からないコードを、どうやって小さく理解し、安全に書き換えるか。**


## なぜ「読む」練習から始めるのか

これからの回では、完成したコードを**少しだけ書き換えて**実験します。ゼロから書けなくても、
**読めて・1か所いじれる**ようになれば十分に前へ進めます。この回は、以後ずっと出てくる
4つの部品（値・リスト・辞書・繰り返し・関数）に絞って読み方を身につけます。文法の網羅はしません。

Copilotは「一発で完成品を作らせる道具」ではなく、「短い相談を何度もする相棒」として使います。
提案は必ず1つずつ試し、出力を自分の目で確かめます。


## 変数・リスト・辞書：データを入れる3つの箱

- **変数**：1つの値に名前を付けた箱（例：`sample_name`）。
- **リスト `[...]`**：順番のある複数の値（例：温度の並び）。
- **辞書 `{key: value}`**：名前で値を引く箱（例：`experiment["solvent"]`で溶媒を取り出す）。

`type(x)`は「その値が何型か」を教えてくれます。実行して、3つの箱の見た目の違いを確かめましょう。


In [ ]:
sample_name = "CMP-0001"
temperatures = [60, 75, 90]
experiment = {"sample_id": sample_name, "solvent": "EtOH", "active": 1}
print(type(sample_name), sample_name)
print(type(temperatures), temperatures)
print(type(experiment), experiment)


### 出力の読み方

- `<class 'str'>`は**文字列**、`<class 'list'>`は**リスト**、`<class 'dict'>`は**辞書**。
- 辞書は`{'sample_id': 'CMP-0001', ...}`のように、**名前（キー）と値**の組で並びます。
- pandasの表（`df`）は、ざっくり言うと「辞書（列名→列の値）」と「リスト（行の並び）」を合わせたものです。この3つが分かると表データも読みやすくなります。


## 演習：`for`（繰り返し）と`if`（条件分岐）を読む

`for`は「リストの要素を1つずつ取り出して同じ処理を繰り返す」書き方、`if ... else`は
「条件で処理を分ける」書き方です。**実行する前に、何行表示されるか予想**してから動かしましょう。
予想と結果を比べるのが、コードを読む力を最短で伸ばすコツです。


In [ ]:
for temperature in temperatures:
    label = "高温条件" if temperature >= 75 else "低温条件"
    print(temperature, label)


### 出力の読み方

- `temperatures`は3要素なので**3行**出ます（予想は合っていましたか？）。
- 各行で`temperature`が60→75→90と変わり、`75以上か`で「高温／低温」が切り替わります。
- `A if 条件 else B`は「条件が真ならA、偽ならB」を1行で書く形。`if:` を複数行で書いても同じ意味です。


## 関数：処理に名前を付けて再利用する

**関数**は「入力を受け取り、決まった処理をして、結果を返す」部品です。同じ計算を何度も書かずに済みます。

- `def 関数名(引数: 型) -> 戻り値の型:` の**型ヒント**は、読み手（と生成AI）への注釈です。動作は変えませんが、誤解を減らします。
- 直後の文字列は**docstring**（関数の説明）。`help(関数)`で読めます。
- `[celsius_to_kelvin(v) for v in temperatures]`は**リスト内包表記**。「各要素に関数をかけた新しいリスト」を1行で作ります。


In [ ]:
def celsius_to_kelvin(celsius: float) -> float:
    "摂氏をケルビンへ変換する。"
    return celsius + 273.15

converted = [celsius_to_kelvin(value) for value in temperatures]
print(converted)
help(celsius_to_kelvin)


### 出力の読み方

- `converted`は、各温度に273.15を足したリスト（例：`[333.15, 348.15, 363.15]`）。
- `help(...)`は、書いておいたdocstringと引数の形を表示します。**自分の関数にも説明が付く**ことを体験しておきましょう。


## 演習：エラーは「読む」もの。省略せず全文を見る

エラーは失敗ではなく、**どこで何が起きたかの手がかり**です。わざと存在しない要素を取り出して、
エラーの形を観察します。`try/except`は「エラーが出ても止まらず、内容を受け取る」書き方です。


In [ ]:
try:
    temperatures[10]
except Exception as error:
    print(type(error).__name__)
    print(error)


### 出力の読み方

- `IndexError`という**エラーの種類（名前）**と、`list index out of range`という**説明**が出ます。
- リストは0番から数えるので、3要素の`temperatures`に`[10]`は存在せず、範囲外エラーになります。
- 実際のエラーでは、**末尾の1〜2行**（種類とメッセージ）にいちばん近い原因が書かれています。Copilotに貼るときも、この全文を省略しないことが大切です。


## 変更して確認

`temperatures`へ温度を1つ追加し、`for`ループと変換結果の表示がどう変わるか確認します。

## Copilotへの相談

気になるセルを貼り、「各行の実行後に、変数の型と中身がどう変わるか表で説明して」と依頼します。
提案は1つずつ試し、必ず出力で答え合わせをします。


## 発展（任意）：テストで守る小さなユーティリティ

ここからは発展です。「実行できる」ことと「正しい」ことは別物です。特にCopilotが書いたコードは、
**普通の入力では動いても、変な入力で静かに間違える**ことがあります。そこで、**変な入力を先に想定して
弾く関数**を書きます。

- `raise TypeError(...)` / `raise ValueError(...)` は、「この入力は受け付けない」と**わざとエラーを起こす**書き方。
- こうしておくと、間違った使い方をした人にすぐ気づいてもらえます（沈黙して誤った答えを返すより安全）。


In [ ]:
def celsius_to_kelvin_checked(celsius: float) -> float:
    "型と物理的な下限を検査してから摂氏をケルビンへ変換する。"
    if not isinstance(celsius, (int, float)):
        raise TypeError("温度は数値で入力してください")
    if celsius < -273.15:
        raise ValueError("絶対零度より低い値は指定できません")
    return celsius + 273.15

for value in [25, -273.15, -300, "25"]:
    try:
        print(value, "->", round(celsius_to_kelvin_checked(value), 2))
    except (TypeError, ValueError) as error:
        print(value, "->", type(error).__name__, error)


### 出力の読み方

4つの入力それぞれの結果が並びます。`25`は正常変換、`-273.15`は境界（絶対零度ちょうど）でOK、
`-300`は物理的にありえないので`ValueError`、`"25"`は数値でなく文字列なので`TypeError`。
**正常・境界・異常**を1度に確かめられました。


### assertで「期待する答え」を先に書いて固定する

`assert 式` は「式が真でなければ止まれ」という自己点検でした。これを使うと、関数の**テスト**が書けます。
コツは、**答えを先に書いてから**関数を作ること。ここではIQR法（四分位範囲）で外れ値を除く関数を、
正常・空リスト・NaN混在の3ケースで検証します。


In [ ]:
import numpy as np

def drop_outliers_iqr(values: list[float], k: float = 1.5) -> list[float]:
    "IQR法で外れ値を除いた値のリストを返す。NaNは事前に除く。"
    clean = [v for v in values if v == v]  # NaN(v != v)を除外
    if not clean:
        return []
    q1, q3 = np.percentile(clean, [25, 75])
    iqr = q3 - q1
    low, high = q1 - k * iqr, q3 + k * iqr
    return [v for v in clean if low <= v <= high]

assert drop_outliers_iqr([10, 11, 12, 13, 1000]) == [10, 11, 12, 13]
assert drop_outliers_iqr([]) == []
assert drop_outliers_iqr([5, 5, float("nan")]) == [5, 5]
print("すべてのテストを通過しました")


### 出力の読み方

- 3つの`assert`がすべて通ると、最後の`print`だけが表示されます。**エラーが出ない＝合格**です。
- もし関数を壊すと（例：`k`を0にする）、どの`assert`で止まったかが表示され、**間違いの場所がすぐ分かります**。
- `v == v`が`False`になるのはNaNだけ、という小技で欠損を除いています。


## 自由課題（任意）：関数の「性質」を調べる

外れ値を除いた後、もう一度同じ関数をかけると、さらに減るでしょうか。1回目で四分位が変わるため、
**必ずしも同じ結果（冪等）にはなりません**。こうした「関数の性質」を意識すると、思わぬ副作用に気づけます。


In [ ]:
rng = np.random.default_rng(0)
sample = rng.normal(50, 5, 200).tolist()
once = drop_outliers_iqr(sample)
twice = drop_outliers_iqr(once)
print("1回適用後の件数:", len(once))
print("2回目でさらに減った件数:", len(once) - len(twice))
print("2回目で変化なし(冪等):", once == twice)


### 出力の読み方

`2回目でさらに減った件数`が0でなければ、この関数は**冪等ではない**（適用回数で結果が変わる）と分かります。
外れ値除去を繰り返し適用する前処理は、この性質のせいで「消しすぎ」が起きやすい、という教訓につながります。


## 追加演習（任意）

90分の外で、以後よく使うPythonの部品をもう少し練習します。飛ばしても本編は進められます。
まずは`enumerate`（番号付き繰り返し）・`zip`（同時に回す）・`sorted`（並べ替え）・条件付き内包表記です。


In [ ]:
samples = ["CMP-0001", "CMP-0002", "CMP-0003"]
yields = [82.5, 40.1, 63.7]

for i, name in enumerate(samples, start=1):        # 番号付きで回す
    print(i, name)

pairs = {name: y for name, y in zip(samples, yields)}   # 2つを同時に回して辞書化
print("辞書:", pairs)

ranked = sorted(pairs.items(), key=lambda kv: kv[1], reverse=True)  # 収率降順
print("収率降順:", ranked)

high = [name for name, y in pairs.items() if y >= 60]   # 条件付き内包表記
print("収率60以上:", high)


### 出力の読み方

- `enumerate`は`(番号, 要素)`を返すので、行番号付きの表示に便利。
- `zip`は複数リストを同時に回します。`for a, b in zip(...)`の形は頻出です。
- `sorted(..., key=..., reverse=True)`で並べ替え。`key`に「何で並べるか」を関数で渡します。
- これらは`for`ループを短く読みやすくする道具で、pandasの内部でも同じ発想が使われています。


### 自分の「状態を持つ部品」を作る（クラス入門）

関数は入力→出力の1回きりですが、**クラス**は状態を持ち続けられます。値を足しながら件数と平均を
保つ小さなクラスを書き、`assert`で動作を確かめます。難しければ「こういう書き方がある」で十分です。


In [ ]:
class RunningStats:
    "値を1つずつ足しながら件数・合計・平均を保つ小さなクラス。"
    def __init__(self):
        self.n = 0
        self.total = 0.0
    def add(self, value: float) -> None:
        self.n += 1
        self.total += value
    @property
    def mean(self) -> float:
        return self.total / self.n if self.n else float("nan")

stats = RunningStats()
for y in [82.5, 40.1, 63.7]:
    stats.add(y)
assert stats.n == 3
assert abs(stats.mean - 62.1) < 0.1
print(f"件数={stats.n} 平均={stats.mean:.1f}")


### 出力の読み方

`add`を呼ぶたびに内部の`n`と`total`が更新され、`mean`はいつでも現在の平均を返します。`assert`が通れば
実装は期待どおり。scikit-learnのモデルも「`fit`で状態を覚え、`predict`で使う」クラスなので、この
仕組みが分かると内部のイメージがつかめます。


### pandasに橋渡しする

第1回パート3で本格的に使うpandasを、ひと足先に少しだけ触ります。CSVを読み、1列（Series）の平均や
種類を取り出します。


In [ ]:
import pandas as pd

data = pd.read_csv(DATA / "compound_experiments.csv")
print("1列の型:", type(data["yield_pct"]).__name__)
print("平均収率:", round(data["yield_pct"].mean(), 1))
print("溶媒の種類:", data["solvent"].dropna().unique().tolist())


### 出力の読み方

`data["yield_pct"]`は1列（Series）で、`.mean()`のような集計をそのまま呼べます。`.unique()`は値の種類、
`.dropna()`は欠損を除く指定。ここまで来れば、第1回パート3のpandasはぐっと読みやすくなります。


---

# パート3：pandasで表データに触る

**このパートの問い：初めて見る表データを受け取ったら、最初に何を見るか。**


## pandasは「表を操る道具」

pandasは、Excelのような表（DataFrame）をPythonで扱うライブラリです。研究データの多くは表なので、
これが読めると分析の8割は前に進みます。この回で身につけるのは、初見の表に対して**同じ手順で
最初の点検をする**習慣です。

初見データを受け取ったら、まず次の4つを見ます：**大きさ（行数×列数）／型（数値か文字か）／
欠損（空欄はどこか）／ばらつき（平均や範囲）**。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


## 演習：表の「健康診断」を1度に行う

次のセルは、点検の4項目をまとめて表示します。`df.shape`で大きさ、`df.dtypes`で型、
`df.isna()`で欠損、`df.describe()`で要約統計。**関数名がそのまま意味**なので、少しずつ覚えられます。


In [ ]:
print("形:", df.shape)
quality = pd.DataFrame({
    "データ型": df.dtypes.astype(str),
    "欠損数": df.isna().sum(),
    "欠損率": df.isna().mean().round(3),
    "ユニーク数": df.nunique(),
})
display(quality)
display(df.select_dtypes(include="number").describe().T.round(2))


### 出力の読み方

- **1つ目の表（品質表）**：各列の型・欠損数・欠損率・値の種類数。`object`は文字列、`float64`/`int64`は数値。欠損率が高い列や、`sample_id`のようにユニーク数＝行数の列（＝ただの名札）に気づけます。
- **2つ目の表（describe）**：数値列の件数・平均・標準偏差・最小/四分位/最大。`temperature_c`の`max`が極端に大きいなど、**怪しい値の当たり**をここで付けます。
- `.T`は表を**転置**（行列入れ替え）して、列がたくさんあっても縦に読めるようにする工夫です。


## 行と列を選ぶ：`loc` と `query`

分析は「必要な部分だけ取り出す」の連続です。2つの基本を覚えます。

- `df.loc[行の条件, 列のリスト]`：**場所を指定して取り出す**。
- `df.query("条件式")`：**条件を文章のように書いて絞り込む**。複数条件（`and`/`or`）が読みやすいのが利点です。


In [ ]:
columns = ["sample_id", "solvent", "catalyst", "temperature_c", "yield_pct", "active"]
display(df.loc[:4, columns])
subset = df.query("catalyst == 'Cat-A' and temperature_c >= 80")[columns]
print("Cat-Aかつ80℃以上:", len(subset), "件")
subset.head()


### 出力の読み方とつまずきポイント

- `df.loc[:4, columns]`は「行番号0〜4」×「指定した6列」。`loc`の範囲指定は**末尾を含む**点がPythonの通常のスライス（末尾を含まない）と違うので注意します。
- `query`の中では、文字列は`'Cat-A'`のように**引用符**で囲みます。列名はそのまま書けます。
- `len(subset)`で、条件に合った件数が分かります。**まず件数を確かめる**のは、絞り込みが意図どおりかの安全確認です。


## 演習：カテゴリごとにまとめて比べる（groupby）

「溶媒ごとの平均収率は？」のような問いには`groupby`が使えます。**同じ値の行をまとめて、
件数・平均・ばらつきなどを一気に計算**します。平均だけでなく**件数（size）とばらつき（std）**も
一緒に見るのが、だまされないコツです。


In [ ]:
solvent_summary = (
    df.groupby("solvent", dropna=False)
      .agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean"),
           収率SD=("yield_pct", "std"), 活性率=("active", "mean"))
      .sort_values("平均収率", ascending=False)
)
solvent_summary.round(2)


### 出力の読み方

- 溶媒ごとに1行、件数・平均収率・収率のばらつき・活性率が並び、平均収率の高い順に並びます。
- **平均が高くても件数が極端に少ない**溶媒は、たまたまかもしれません。件数の小さい行の平均は割り引いて読みます。
- `dropna=False`にしているので、溶媒が欠損の行も1グループとして見えます（欠損を見逃さない工夫）。


## 変更して確認

`groupby("solvent")`を`"catalyst"`や`"scaffold_group"`へ変えて、順位がどう変わるか見ます。
順位が変わる理由は、**データだけから断定せず仮説として**書き留めます（第4〜5回でその検証を学びます）。


## 発展（任意）：多軸集計・処理の連結・速度

発展として、実務でよく使う3つを扱います。**pivot_table**（2軸のクロス集計）、
**pipe**（処理を関数でつなぐ）、そして**ベクトル化**（速く書く）です。


### pivot_table：2つの軸で同時に集計する

「触媒×溶媒」のように2軸で平均を見たいときは`pivot_table`が便利です。Excelのピボットテーブルと
同じ発想で、`index`（縦軸）・`columns`（横軸）・`values`（集計する値）・`aggfunc`（集計方法）を指定します。


In [ ]:
pivot = pd.pivot_table(df, index="catalyst", columns="solvent", values="yield_pct", aggfunc=["count", "mean"])
pivot.round(1)


### 出力の読み方

行が触媒、列が溶媒で、各マスに「件数」と「平均収率」が入ります。件数が0や極端に少ないマスは、
平均が空欄や不安定になります。**組み合わせによって効き方が変わる**様子（交互作用）の当たりを付けられます。


### pipe：処理を「関数の流れ」としてつなぐ

複数の加工を続けるとき、中間変数を増やすと読みにくくなります。`.pipe(関数)`を使うと、
**表を関数に通して次へ渡す**流れを、上から下へ素直に書けます。元データを壊さないよう、関数内で`copy()`します。


In [ ]:
def add_quality_flags(frame):
    "収率の中央値以上かどうかのフラグ列を足して返す（元は変更しない）。"
    out = frame.copy()
    out["high_yield"] = out["yield_pct"] >= out["yield_pct"].median()
    return out

summary = (
    df
    .pipe(add_quality_flags)
    .groupby(["catalyst", "high_yield"], observed=True)
    .agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean"))
    .round(2)
)
summary


### 出力の読み方

触媒×「高収率かどうか」で件数と平均収率が出ます。**加工（フラグ付け）→集計**という流れが、
1つの縦長の式で読める点に注目してください。処理が増えても`.pipe(...)`を足すだけで拡張できます。


## 自由課題（任意）：`apply`と「ベクトル化」の速度差

同じ判定を2通りで書き、時間を比べます。行を1つずつ処理する`apply`は読みやすい一方、遅くなりがち。
列全体へ一括で演算する**ベクトル化**は速く、pandasの本領です。


In [ ]:
import time

def slow_flag(frame):
    return frame.apply(lambda r: r["temperature_c"] >= 80 and r["catalyst"] == "Cat-A", axis=1)

def fast_flag(frame):
    return (frame["temperature_c"] >= 80) & (frame["catalyst"] == "Cat-A")

t0 = time.perf_counter(); a = slow_flag(df); t1 = time.perf_counter()
b = fast_flag(df); t2 = time.perf_counter()
print("apply     :", round((t1 - t0) * 1000, 2), "ms")
print("vectorized:", round((t2 - t1) * 1000, 2), "ms")
print("結果一致:", bool((a.fillna(False) == b.fillna(False)).all()))


### 出力の読み方

- 2つの時間（ミリ秒）を比べると、**ベクトル化の方が速い**はずです。420行では差は小さくても、数十万行では体感が大きく変わります。
- `結果一致: True`は、2つの書き方が**同じ答え**を出した確認。速く書いても結果が同じであることを、必ず検証します。
- 教訓：`apply`が必要な場面もありますが、まず「列演算で書けないか」を考える習慣が、速く読みやすいコードにつながります。


## 追加演習（任意）

実データで頻出のpandas操作を、もう少し重めに練習します。90分の外の自習向けです。
まずは**度数の集計**（`value_counts`）と**クロス集計**（`crosstab`）です。


In [ ]:
print(df["catalyst"].value_counts(dropna=False))
print()
display(pd.crosstab(df["catalyst"], df["solvent"]))


### 出力の読み方

- `value_counts`は各カテゴリの件数を多い順に。`dropna=False`で欠損も1カテゴリとして数えます。
- `crosstab`は2つのカテゴリの組み合わせ件数の表。どの触媒×溶媒の組が多い／少ないかが一望できます。件数の少ない組は、後の分析で平均が不安定になりやすい箇所です。


### 日付を扱う（datetime）

`experiment_date`は文字列です。`pd.to_datetime`で日付型に変えると、月ごとの集計や期間の計算ができます。
時系列の分割（第2回パート3）にもつながる大事な操作です。


In [ ]:
dated = df.copy()
dated["experiment_date"] = pd.to_datetime(dated["experiment_date"])
dated["month"] = dated["experiment_date"].dt.to_period("M").astype(str)
monthly = dated.groupby("month").agg(件数=("sample_id", "size"), 平均収率=("yield_pct", "mean")).round(1)
display(monthly.head(6))


### 出力の読み方

月ごとの件数と平均収率が並びます。`.dt.to_period("M")`で「年月」に丸めています。実データでは、
月やバッチで性能が変わることがあり、こうした時間軸の集計が異常検知や分割設計の入口になります。


### 表をつなぐ（merge）と、形を変える（melt）

`merge`は2つの表をキーで結合します。ここでは「触媒ごとの平均収率」を各行に付け直し、
各試料が平均より上か下かを計算します。`melt`は横広の表を縦長へ変える操作です。


In [ ]:
group_mean = df.groupby("catalyst")["yield_pct"].mean().rename("触媒平均収率").reset_index()
merged = df[["sample_id", "catalyst", "yield_pct"]].merge(group_mean, on="catalyst")
merged["平均との差"] = (merged["yield_pct"] - merged["触媒平均収率"]).round(1)
display(merged.head())

wide = df.head(3)[["sample_id", "molecular_weight", "logp", "tpsa"]]
long = wide.melt(id_vars="sample_id", var_name="記述子", value_name="値")
display(long)


### 出力の読み方

- **merge後**：各試料に「触媒平均収率」列が付き、「平均との差」で相対評価ができます。この「群平均を特徴量にする」発想は第4回パート2のtarget encodingにつながります（ただしリークに注意）。
- **melt後**：3列だった記述子が「記述子・値」の2列に畳まれ、行数が増えます。可視化ライブラリはこの縦長形式を好むことが多いです。


---

# パート4：分布・欠損・外れ値を確認する

**このパートの問い：モデルを作る前に、データの怪しいところをどう見つけるか。**


## EDA＝モデルを作る前にデータをよく見る工程

EDA（探索的データ分析）は、いきなりモデルを作らず、まずデータをよく見る工程です。目的は
「きれいなグラフを作ること」ではなく、**モデルを惑わせる怪しい点（偏り・欠損・外れ値）を先に見つけ、
検証できる仮説を作ること**です。

見る順番にはコツがあります：**1変数（分布）→ 2変数（関係）→ 群別（カテゴリごと）**。
いきなり複雑な図に行かず、単純な図から積み上げます。次のセルはまず描画の下準備（日本語表示と
見た目のテーマ設定）です。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
# グラフの日本語が文字化けしないようにする設定です。中身は今は理解しなくてOK、そのまま実行してください。
import matplotlib.pyplot as plt
from matplotlib import font_manager
for _name in ["Yu Gothic", "Meiryo", "Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]:
    if _name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _name
        break
import seaborn as sns
sns.set_theme(style="whitegrid")


## 演習：1変数の分布を見る（ヒストグラムと箱ひげ図）

まず1列ずつ「値がどこに、どれだけあるか」を見ます。

- **ヒストグラム**：値を区間に分け、各区間の件数を棒で表す。山の形・偏り・飛び離れた値が見えます。
- **箱ひげ図**：中央値・四分位・外れ値候補（ひげの外の点）をコンパクトに表す。外れ値探しに向きます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="yield_pct", bins=20, ax=axes[0])
axes[0].set_title("収率の分布")
sns.boxplot(data=df, x="reaction_time_h", ax=axes[1])
axes[1].set_title("反応時間：外れ値候補を探す")
plt.tight_layout()


### 出力の読み方

- **左（収率のヒストグラム）**：山が1つか2つか、左右どちらに裾を引くかを見ます。裾が長い＝一部に極端な値。
- **右（反応時間の箱ひげ図）**：箱が中央50%、ひげの外の点が外れ値候補。**右端にぽつんと離れた点**があれば、それが要調査の試料です（このデータには意図的に極端な値を仕込んであります）。
- まだ「削除」はしません。EDAは**見つける**段階です。


## 2変数の関係とカテゴリ比較（散布図・箱ひげ図）

次に「2つの列の関係」を見ます。散布図は連続値どうしの関係、色分け（`hue`）で3つ目の情報（触媒）も
重ねられます。カテゴリごとの違いは箱ひげ図で比べます。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=df, x="temperature_c", y="yield_pct", hue="catalyst", alpha=0.65, ax=axes[0])
axes[0].set_title("温度と収率")
sns.boxplot(data=df, x="catalyst", y="yield_pct", ax=axes[1])
axes[1].set_title("触媒別の収率")
plt.tight_layout()


### 出力の読み方

- **左（温度×収率）**：右肩上がりの直線ではなく、**中くらいの温度で収率が高くなる山型**に見えるはずです。「関係＝直線」とは限らないことを、目で確認しておきます（第2回パート1の発展で扱う相互情報量につながります）。
- 点の色（触媒）で**かたまり**ができていれば、触媒が収率に効いている手がかり。
- **右（触媒別の箱ひげ）**：触媒ごとに箱の高さ（収率の中心）が違えば、触媒の効果が疑われます。ただし件数が少ない触媒は割り引いて読みます。


## 演習：欠損と「明らかに怪しい値」を表で押さえる

図で当たりを付けたら、表で具体的に特定します。どの列にいくつ欠損があるか、そして温度が異常に
大きい上位5件を実際に取り出します。


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("欠損数"))
display(df.nlargest(5, "temperature_c")[["sample_id", "temperature_c", "reaction_time_h", "yield_pct"]])


### 出力の読み方

- **1つ目の表**：欠損のある列と件数。欠損の多い列は、後で「埋める／落とす／別扱い」の判断が要ります。
- **2つ目の表**：温度が高い順の5件。`180.0`のような**周囲から突出した値**があれば、入力ミスか特殊な実験かを疑い、`sample_id`を控えて確認先を考えます。
- ここでも即削除しないのが鉄則。**「誰に確認するか」「残した場合に何が起きるか」**まで考えてから対処します。


## 変更して確認

散布図の色分け（`hue`）を`catalyst`から`solvent`へ変え、見え方の違いを1つ挙げます。

## 注意

外れ値＝入力ミス、ではありません。本物の珍しい現象のこともあります。EDAの結論は「削除」ではなく、
**「確認すべき仮説」**の形で残します。


## 発展（任意）：印象を統計量で裏づける

図の印象は主観的です。ここでは4つの道具で客観化します：**相関**（直線的な関係）、
**相互情報量**（曲がった関係も拾う）、**欠損機構**（欠損の起こり方）、**多変量外れ値**（組み合わせの異常）。


### 相関ヒートマップ：全列の関係を一望する

`corr()`は数値列すべての**相関係数**（-1〜+1）を計算します。+1に近いほど一緒に増え、-1に近いほど
片方が増えると片方が減る関係。色の濃淡で一望できます。ただし**相関は「直線的な」関係しか測れない**
ことに注意します。


In [ ]:
numeric_cols = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa", "yield_pct"]
correlation = df[numeric_cols].corr()
plt.figure(figsize=(8, 5))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("数値列の相関（因果ではない）")
plt.tight_layout()


### 出力の読み方

- 対角線は自分自身との相関で必ず1.00。赤いマスほど正、青いマスほど負の相関。
- `temperature_c`と`yield_pct`の相関は**意外と弱い**はずです（山型の関係なので直線相関では捉えきれない）。
- **重要**：相関は因果ではありません。「AとBが一緒に動く」ことと「AがBの原因」は別物です。


### 相互情報量：曲がった関係も拾う

温度のように「最適点で収率が最大」の山型は、相関では弱く見えます。**相互情報量**は直線に限らず
「片方を知るともう片方の予想がどれだけ絞れるか」を測るので、こうした関係を拾えます。相関と並べて読みます。


In [ ]:
from sklearn.feature_selection import mutual_info_regression

mi_source = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
mi_frame = df[mi_source + ["yield_pct"]].dropna()
mi = mutual_info_regression(mi_frame[mi_source], mi_frame["yield_pct"], random_state=42)
pearson = mi_frame[mi_source].corrwith(mi_frame["yield_pct"]).abs()
compare = pd.DataFrame({"相互情報量": mi, "|相関|": pearson.to_numpy()}, index=mi_source)
compare.sort_values("相互情報量", ascending=False).round(3)


### 出力の読み方

`temperature_c`は**相互情報量は大きいのに|相関|は小さい**、という食い違いが見えるはずです。これが
「相関だけで特徴量を捨ててはいけない」理由です。両方を見て、関係の形は散布図で確かめます。


### 欠損の起こり方（欠損機構）を疑う

欠損はランダムとは限りません。**MCAR**（完全にランダム）、**MAR**（他の列で説明できる偏り）、
**MNAR**（値そのものに依存）で対処が変わります。ここでは「温度の欠損率が溶媒で偏るか」を見ます。


In [ ]:
miss = df.assign(temp_missing=df["temperature_c"].isna())
by_solvent = miss.groupby("solvent", dropna=False)["temp_missing"].mean().round(3)
print("溶媒別の温度欠損率:")
print(by_solvent)
print("溶媒でほぼ一定ならMCARに近い。偏るならMARを疑う。")


### 出力の読み方

溶媒によって温度欠損率が大きく違えば、欠損は溶媒と関係している（MARの疑い）＝
「一律に中央値で埋める」のが危ういサインです。値がほぼ一定なら、単純な補完でも大きな害は出にくいと判断できます。


### 多変量外れ値：組み合わせの異常を探す

「温度は普通、時間も普通、でもその組み合わせは他にない」という試料は、1列ずつ見ても見つかりません。
`IsolationForest`は**複数列を同時に見て、周囲から孤立した点**を外れ値候補として検出します。


In [ ]:
from sklearn.ensemble import IsolationForest

iso_cols = ["temperature_c", "reaction_time_h", "concentration_m", "yield_pct"]
iso_data = df[iso_cols].fillna(df[iso_cols].median())
flags = IsolationForest(contamination=0.03, random_state=42).fit_predict(iso_data)
outliers = df.loc[flags == -1, ["sample_id", *iso_cols]]
print("多変量外れ値候補:", len(outliers), "件")
outliers.round(2)


### 出力の読み方

- `contamination=0.03`は「全体の約3%を外れ値候補とみなす」設定です（多すぎ・少なすぎると感じたら調整）。
- 出た試料を1件ずつ見て、**どの列の組み合わせが変か**を考えます。単変量の箱ひげ図では正常だった試料が混じっていれば、多変量で見る価値があった、ということです。
- ここでも自動削除はせず、確認対象のリストとして扱います。


## 追加演習（任意）

可視化の引き出しを増やします。90分の外の自習向けです。まず**pairplot**で、複数の数値列の関係を
一度に俯瞰します（散布図と分布のマトリクス）。


In [ ]:
subset = df[["temperature_c", "reaction_time_h", "yield_pct", "catalyst"]].dropna()
sns.pairplot(subset, hue="catalyst", corner=True, plot_kws={"alpha": 0.5})


### 出力の読み方

対角線は各列の分布、対角線以外は2列の散布図で、色は触媒。**触媒ごとにかたまりができている**列の組が
あれば、それが効いている手がかり。多くの列を一気に眺めて当たりを付け、気になったペアを個別の図で
深掘りします（俯瞰→詳細の順）。


### バイオリン図とカウント図

**バイオリン図**は箱ひげ図より分布の形（山が1つか2つか）が分かります。**カウント図**はカテゴリの件数。
分布の形と件数を押さえると、平均の解釈がぐっと安全になります。


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.violinplot(data=df, x="catalyst", y="yield_pct", ax=axes[0])
axes[0].set_title("触媒別の収率分布（バイオリン）")
sns.countplot(data=df, x="solvent", ax=axes[1])
axes[1].set_title("溶媒の件数")
plt.tight_layout()


### 出力の読み方

- **バイオリン**：横幅が太い高さに値が集まっています。二山（2つのふくらみ）なら、隠れた別グループの存在を疑います。
- **カウント図**：件数の少ない溶媒は、以降の群別集計で平均が不安定になりやすい箇所。分析前に把握しておきます。


### 群別の目的変数と、目的変数との関連を棒で見る

「触媒別の活性率」と「収率との|相関|が強い列」を棒グラフで並べます。EDAの締めとして、
**目的変数（active/yield）に効きそうな列**の当たりを付けます。


In [ ]:
active_rate = df.groupby("catalyst")["active"].mean().sort_values(ascending=False)
corr_target = df.select_dtypes("number").corr()["yield_pct"].drop("yield_pct").abs().sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
active_rate.plot.bar(ax=axes[0], title="触媒別の活性率")
corr_target.plot.bar(ax=axes[1], title="収率との|相関|")
plt.tight_layout()


### 出力の読み方

- **左**：触媒によって活性率が違えば、触媒は分類（active）に効く候補。
- **右**：収率との|相関|が高い列が、回帰（yield）で効く候補。ただし第2回パート1で確認したとおり、相関が低くても相互情報量が高い列（温度など）を見落とさないよう、相関の棒だけで判断しないこと。
- ここで挙がった候補列が、第2回パート2以降の特徴量選びの出発点になります。


---

## よくある誤り

- カーネルの選択で別のPythonを選んでしまう
- uvとCondaを同じもの・同じ目的だと思い込む
- Gitを使わないと勉強会に参加できないと思い込む
- Notebookを途中から実行して変数がない
- Copilotの長い修正を一度に採用する
- エラー全文を読まずにセルを繰り返し実行する
- 列の単位や定義を確認せず計算する
- 行ごとのapplyを多用して遅く読みにくくする
- 件数が極端に少ない群の平均を強く信じる
- 外れ値を自動削除する
- 相関を因果と読む
- 見栄えの良い図だけを選ぶ

## 自習（任意・30〜60分）

- pyproject.tomlとuv.lockを開き、どのライブラリがどのバージョンで固定されているか3つ挙げる
- docs/environment-and-git-basics.mdを読み、Gitのclone/pullを実際に試す
- 収率のリストから外れ値をIQRで除く関数を、型ヒントとテスト付きで書く
- 正常値・空リスト・NaN混在の3ケースを、期待結果を先に書いてから検証する
- 触媒×溶媒の件数・平均収率・標準偏差をpivot_tableで作る
- applyとベクトル化の実行時間を比較し、差をm%で記録する
- 相互情報量の上位3列について、散布図で関係の形を確認する
- IsolationForestの外れ値候補2件を、残す場合と除く場合で整理する

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 仮想環境を分ける理由は何か
2. uvとCondaの違いは何か
3. コミットとプッシュはそれぞれ何をする操作か
4. 型ヒントとdocstringは何の役に立つか
5. assertは何を保証し、何を保証しないか
6. 生成AIのコードを何で確認するか
7. shapeの2つの数は何か
8. method chainingの利点と注意点は何か
9. applyよりベクトル化を選ぶ理由は何か
10. 相関係数と相互情報量はどう違うか
11. MCARとMARの違いは何か
12. 多変量外れ値が単変量で見つからない理由は何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
